# Function Calling with OpenAI
Defines a JSON-schema-based tool and lets the LLM invoke it automatically.


In [28]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv("/home/amir/source/.env")

WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [36]:
from openai import OpenAI
import requests
import os

client = OpenAI(api_key=OPENAI_API_KEY)

# Define the tool schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a given location using WeatherAPI.com",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City name (e.g. 'London')"
                    }
                },
                "required": ["location"]
            }
        }
    }
]


In [45]:
# LLM call with function support
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "What's the weather like in London?"}],
    tools=tools,
    tool_choice="auto"
)

# Extract function call arguments
for choice in response.choices:
    if choice.message.tool_calls:
        args = choice.message.tool_calls[0].function.arguments
        print("Function call arguments:", args)


Function call arguments: {"location":"London"}


In [46]:
# Function that calls WeatherAPI.com
def get_current_weather(location: str) -> dict:
    # call example http://api.weatherapi.com/v1/current.json?key={your_WEATHERAPI_KEY}&q=London&aqi=no
    print(f"🌍 Fetching weather for: {location}")
    url = "http://api.weatherapi.com/v1/current.json"
    params = {
        "key": WEATHER_API_KEY,
        "q": location,
        "aqi": "no"
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        current = data["current"]
        return {
            "location": data["location"]["name"],
            "region": data["location"]["region"],
            "country": data["location"]["country"],
            "temperature_c": current["temp_c"],
            "condition": current["condition"]["text"],
            "wind_kph": current["wind_kph"]
        }
    else:
        return {"error": f"API Error ({response.status_code}): {response.text}"}


In [47]:
# Send user message to LLM
response = client.chat.completions.create(
    model="gpt-5.4",
    messages=[
        {"role": "user", "content": "What's the weather in the capital of Israel?"}
    ],
    tools=tools,
    tool_choice="auto",
)
print("LLM Response:", response)
print("Tool calls:", response.choices[0].message.tool_calls)

LLM Response: ChatCompletion(id='chatcmpl-DcaU4o6Bm7pshkjObTEXKKA4UYVTd', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ebQ9bqbLcyuhaNahRxoJN8SZ', function=Function(arguments='{"location":"Jerusalem"}', name='get_current_weather'), type='function')]))], created=1778089080, model='gpt-5.4-2026-03-05', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=19, prompt_tokens=151, total_tokens=170, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_ebQ9bqbLcyuhaNahRxoJN8SZ', function=Function(argument

In [48]:
# Handle tool calls from LLM
import json

tool_calls = response.choices[0].message.tool_calls
if tool_calls:
    for call in tool_calls:
        print(call)
        # safely parse JSON arguments
        args = json.loads(call.function.arguments)
        if call.function.name == "get_current_weather":
            result = get_current_weather(**args)
            if "error" in result:
                print("⚠️", result["error"])
            else:
                print(f"🌤️ {result['location']}, {result['country']}: "
                      f"{result['temperature_c']}°C, {result['condition']} "
                      f"(Wind: {result['wind_kph']} kph)")
        else:
            result = {"error": "Unknown tool"}

    # Build assistant message containing the original tool call (include id)
    tc0 = response.choices[0].message.tool_calls[0]
    assistant_msg = {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": tc0.id,
                "function": {
                    "name": tc0.function.name,
                    "arguments": tc0.function.arguments,
                },
                "type": "function",
            }
        ]
    }

    # Send the tool result back as a `tool` role message (must include tool_call_id)
    tool_msg = {
        "role": "tool",
        "name": tc0.function.name,
        "content": json.dumps(result),
        "tool_call_id": tc0.id,
    }

    followup = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "user", "content": "What's the weather in the capital of Israel?"},
            assistant_msg,
            tool_msg,
        ],
    )
    print("Assistant (followup):", getattr(followup.choices[0].message, 'content', None))
else:
    print("💬 No function was called. Response:", response.choices[0].message.content)


ChatCompletionMessageFunctionToolCall(id='call_ebQ9bqbLcyuhaNahRxoJN8SZ', function=Function(arguments='{"location":"Jerusalem"}', name='get_current_weather'), type='function')
🌍 Fetching weather for: Jerusalem
🌤️ Jerusalem, Israel: 15.0°C, Mist (Wind: 18.7 kph)
Assistant (followup): In Jerusalem, Israel, it’s currently 15°C with mist and winds around 18.7 kph.


# ChatOpenAI + `initialize_agent` — LangChain Agent

LangChain provides a higher-level abstraction over the OpenAI API, allowing you to build agents that can use tools and perform **multi-step reasoning** with minimal boilerplate.

## The `@tool` decorator

- OpenAI Python SDK that uses a more Pythonic and concise way to define tools for function calling, replacing the manual JSON schema approach.

---

## Key Benefits

- **Agent logic is built-in:**
  - Parses the model’s tool calls automatically.
  - Executes the appropriate Python functions without manual parsing.
  - Sends results back to the LLM for further reasoning or additional tool calls.

- **Automatic chaining** — the model can decide to call multiple tools in sequence  
  *(e.g., get a city → fetch its weather).* 

- **Verbose debugging** with `verbose=True` to see the reasoning process.

---

## How It Works

1. **User prompt:**  
   `"What's the weather in the capital of Israel?"`

2. **Model reasoning:**
   - Calls `get_capital("Israel")` → returns `"Jerusalem"`.
   - Calls `get_current_weather("Jerusalem")`.

3. **Agent orchestration:**
   - Automatically parses tool call requests from the LLM.
   - Runs the matching Python function(s).
   - Sends results back to the model if needed for additional reasoning.

4. **Final answer:**  
   Returned to the user without needing to manually parse or chain calls.

---

## Why Use This Instead of `client.chat.completions.create`?

| Feature | Direct API Call | LangChain Agent |
|---|---|---|
| Tool call parsing | Manual | Automatic |
| Multi-step reasoning | You implement | Built-in |
| Code length | Short for single call | Short for multi-step workflows |
| Flexibility | Maximum control | Quick prototyping |
| Debugging | Manual print/logging | `verbose=True` built-in |


In [43]:
# pip install -U langchain langchain-openai requests

import os, requests
from typing import Dict
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")

@tool
def get_capital(country: str) -> str:
    """Return the capital city for a given country."""
    return {"Israel": "Jerusalem"}.get(country, "Unknown")

@tool
def get_current_weather(location: str) -> Dict:
    """Get the current weather for a given location using WeatherAPI.com."""
    url = "http://api.weatherapi.com/v1/current.json"
    resp = requests.get(
        url,
        params={"key": WEATHER_API_KEY, "q": location, "aqi": "no"},
        timeout=15
    )
    if resp.status_code != 200:
        return {"error": f"API Error ({resp.status_code}): {resp.text}"}

    d = resp.json()
    c = d["current"]

    return {
        "location": d["location"]["name"],
        "country": d["location"]["country"],
        "temperature_c": c["temp_c"],
        "condition": c["condition"]["text"],
        "wind_kph": c["wind_kph"],
    }

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=OPENAI_API_KEY)

agent = initialize_agent(
    tools=[get_capital, get_current_weather],
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True,
)

print(agent.run("What's the weather in the capital of Israel?"))




> Entering new AgentExecutor chain...

Invoking: `get_capital` with `{'country': 'Israel'}`


Jerusalem
Invoking: `get_current_weather` with `{'location': 'Jerusalem'}`


{'location': 'Jerusalem', 'country': 'Israel', 'temperature_c': 15.0, 'condition': 'Mist', 'wind_kph': 18.7}The current weather in Jerusalem, the capital of Israel, is misty with a temperature of 15°C. The wind speed is 18.7 km/h.

> Finished chain.
The current weather in Jerusalem, the capital of Israel, is misty with a temperature of 15°C. The wind speed is 18.7 km/h.
